# Como encontrar URLs de OPeNDAP de NASA Earthdata

Este tutorial demuestra cómo encontrar URLs OPeNDAP desde el [Common Metadata Repository](https://cmr.earthdata.nasa.gov/search) (CMR). El CMR es la API de NASA Earthdata para consultar datasets disponibles a través de muchos servicios de descarga y subconjunto, incluido OPeNDAP. La [API de CMR](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html) es compleja y amplia, y con `pydap.client.get_cmr_urls` las personas usuarias pueden consultar y recuperar URLs OPeNDAP.

**Requisitos para ejecutar este notebook**
1. Tener una cuenta Earth Data Login
2. Conocer el Collection Concept ID (CCID), o Digital Object Identifier (DOI), de la colección de interés.

```{note}
Desde la perspectiva de NASA, una colección es un dataset (a diferencia de un gránulo, que puede entenderse como un archivo individual y que, en conjunto con otros, describe la colección). Por ello, el CCID o DOI son identificadores únicos de ese dataset archivado.
```
**Objetivos**

Usar [PyDAP](https://pydap.github.io/pydap/) para **descubrir todas las URLs opendap en dos casos de estudio simples**:

1. Descubrir todas las URLs OPeNDAP posibles asociadas con un Collection Concept ID específico (y DOI).
2. Descubrir todas las URLs OPeNDAP de una colección que **coincidan con un rango temporal y una caja espacial de interés**. Estos parámetros, y otros, son ampliamente usados por CMR (y Earthdata Search) para filtrar el número de resultados posibles al consultar el CMR, reduciendo así la búsqueda.


`Autor`: Miguel Jimenez-Urias, '25


In [ ]:
from pydap.client import get_cmr_urls
import pydap
import datetime as dt

In [ ]:
print("pydap version: ", pydap.__version__)

## 1) Descubrir datos diarios de clorofila a 4 km desde PACE (Nivel 3)

En este ejemplo nos interesa recuperar TODAS las URLs de gránulos desde OPeNDAP asociadas con una colección de PACE. Para esta colección, el CMR devuelve varias versiones de los datos con respecto a la siguiente variable:

- Gridded Chlorophyll A, Version 3.1


In [ ]:
PACE_ccid = "C3620140255-OB_CLOUD" # <--- This concept collection ID can be found of the Mission page for PACE.

In [ ]:
urls = get_cmr_urls(ccid=PACE_ccid, limit=1000) # limit by default = 50

In [ ]:
urls[:10]

### Identificar los archivos de interés
No todas las URLs anteriores se pueden agregar en una sola colección. Aunque describen las mismas variables, estas estan interpoladas sobre distintos rangos temporales. Queremos datos diarios a resolución de 4 km. Como esta información está codificada en la URL, podemos usar una list comprehension para filtrar más nuestros resultados.


In [ ]:
pace_urls = [url for url in urls if '4km' in url and "DAY" in url]
pace_urls[:4]

In [ ]:
print("We found ", len(pace_urls), " relevant OPeNDAP urls from PACE")

## 2) Acceder a datos swath de ECOSTRESS (datos Nivel 2)

En este ejemplo filtraremos el resultado del CMR para devolver solo URLs de gránulos que tengan datos en un área específica de interés.

- Land Surface Temperature
- Swath de datos durante marzo de 2025, en una caja delimitadora definida abajo:

```python
bounding_box = [-128.847656,41.112469,-107.050781,46.679594]
```

```{note}
Puedes usar una aplicación web, como [bbox finder](http://bboxfinder.com), [geojson](https://geojson.io) o [Earthdata Search](https://search.earthdata.nasa.gov), para construir polígonos. Ten en cuenta que la API de CMR requiere, para una caja delimitadora, el siguiente patrón: [West_Longitude, South_Latitude, East_Longitude, North_Latitude]
```


In [ ]:
ECOSTRESS_ccid = "C2076114664-LPCLOUD"
bounding_box = [-128.847656,41.112469,-107.050781,46.679594]
time_range = [dt.datetime(2025, 3, 1), dt.datetime(2025, 3, 31)]

In [ ]:
urls = get_cmr_urls(ccid=ECOSTRESS_ccid, bounding_box=bounding_box, time_range=time_range, limit=500)
print("Found ", len(urls), "relevant opendap urls for ECOSTRESS data")

In [ ]:
urls[:6]

```{warning}
El CMR devuelve URLs OPeNDAP con datos que caen dentro de nuestra caja delimitadora. Sin embargo, todavía no se ha realizado ningún subconjunto. Para descargar solo los datos que caen dentro del área de interés, la persona usuaria aún necesita subdividir los datos dentro de cada archivo. CMR solo filtra por metadatos. Es decir, por ejemplo, en uno de esos archivos remotos puede haber solo un punto de datos dentro de la caja delimitadora, mientras que en otro archivo remoto es posible que la mayoría de los datos caigan dentro de la caja delimitadora.
```
